# Kaggle Showcase Inference - Plant Classification

Notebook nay dung de:
- load `best_model.pth`
- load 1 anh bat ky
- du doan class cua cay
- hien thi top-k predictions de demo

In [ ]:
from pathlib import Path

INPUT_ROOT = Path('/kaggle/input')
WORKING_DIR = Path('/kaggle/working')

def find_dataset_dir(dataset_name='dataset_plant_classification'):
    direct_candidates = [
        INPUT_ROOT / 'plant_classification' / dataset_name,
        INPUT_ROOT / dataset_name,
    ]
    for candidate in direct_candidates:
        if candidate.exists():
            return candidate

    for candidate in sorted(INPUT_ROOT.rglob(dataset_name)):
        if candidate.is_dir():
            return candidate

    raise FileNotFoundError(f'Could not find {dataset_name} under {INPUT_ROOT}')

def find_model_root(output_dir_name='plant_training_outputs'):
    direct_candidates = [
        INPUT_ROOT / 'best-models' / output_dir_name,
        INPUT_ROOT / output_dir_name,
    ]
    for candidate in direct_candidates:
        if candidate.exists():
            return candidate

    for candidate in sorted(INPUT_ROOT.rglob(output_dir_name)):
        if candidate.is_dir():
            return candidate

    raise FileNotFoundError(f'Could not find {output_dir_name} under {INPUT_ROOT}')

DATASET_DIR = find_dataset_dir()
MODEL_ROOT = find_model_root()

# Chon 1 trong 3 model da train
MODEL_NAME = 'resnet50_fine_tuning'

# Doi duong dan nay thanh anh ban muon demo
# Co the la anh trong dataset hoac anh ban upload len Kaggle notebook session
IMAGE_PATH = DATASET_DIR / 'Tomato'
TOP_K = 5
IMAGE_SIZE = 224

BEST_MODEL_PATH = MODEL_ROOT / MODEL_NAME / 'best_model.pth'
print('DATASET_DIR =', DATASET_DIR)
print('MODEL_ROOT =', MODEL_ROOT)
BEST_MODEL_PATH

In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from PIL import Image
from torchvision import models, transforms
from torchvision.models import ResNet50_Weights

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
class_dirs = sorted([p for p in DATASET_DIR.iterdir() if p.is_dir()])
class_names = [p.name for p in class_dirs]
num_classes = len(class_names)

print('Total classes:', num_classes)
print('First 10 classes:', class_names[:10])

In [ ]:
class CustomCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d((1, 1)),
        )
        self.classifier = nn.Sequential(
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        return self.classifier(x)

def build_resnet50_feature_extractor(num_classes):
    model = models.resnet50(weights=None)
    for param in model.parameters():
        param.requires_grad = False
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_features, num_classes),
    )
    return model

def build_resnet50_finetune(num_classes):
    model = models.resnet50(weights=None)
    for param in model.parameters():
        param.requires_grad = False
    for param in model.layer4.parameters():
        param.requires_grad = True
    for param in model.fc.parameters():
        param.requires_grad = True
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_features, num_classes),
    )
    return model

def build_model(model_name, num_classes):
    if model_name == 'custom_cnn':
        return CustomCNN(num_classes)
    if model_name == 'resnet50_feature_extraction':
        return build_resnet50_feature_extractor(num_classes)
    if model_name == 'resnet50_fine_tuning':
        return build_resnet50_finetune(num_classes)
    raise ValueError(f'Unknown MODEL_NAME: {model_name}')

In [ ]:
transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

model = build_model(MODEL_NAME, num_classes)
state_dict = torch.load(BEST_MODEL_PATH, map_location=DEVICE)
model.load_state_dict(state_dict)
model = model.to(DEVICE)
model.eval()

print('Loaded model:', MODEL_NAME)
print('Weights:', BEST_MODEL_PATH)

In [ ]:
VALID_EXTS = {'.jpg', '.jpeg', '.png'}

if IMAGE_PATH.is_dir():
    candidates = sorted([p for p in IMAGE_PATH.iterdir() if p.is_file() and p.suffix.lower() in VALID_EXTS])
    assert candidates, f'Khong tim thay anh trong folder: {IMAGE_PATH}'
    image_path = random.choice(candidates)
else:
    image_path = IMAGE_PATH

image = Image.open(image_path).convert('RGB')
tensor = transform(image).unsqueeze(0).to(DEVICE)

print('Using image:', image_path)

In [ ]:
with torch.no_grad():
    logits = model(tensor)
    probs = torch.softmax(logits, dim=1)[0]
    top_probs, top_indices = torch.topk(probs, k=min(TOP_K, num_classes))

top_probs = top_probs.detach().cpu().numpy()
top_indices = top_indices.detach().cpu().numpy()

predicted_class = class_names[int(top_indices[0])]
predicted_confidence = float(top_probs[0])

print('Predicted class:', predicted_class)
print('Confidence:', f'{predicted_confidence * 100:.2f}%')

In [ ]:
plt.figure(figsize=(7, 7))
plt.imshow(image)
plt.title(f'Prediction: {predicted_class} ({predicted_confidence * 100:.2f}%)')
plt.axis('off')
plt.show()

In [ ]:
topk_rows = []
for idx, prob in zip(top_indices, top_probs):
    topk_rows.append({
        'class_name': class_names[int(idx)],
        'probability': float(prob),
        'probability_percent': float(prob) * 100,
    })

topk_rows

In [ ]:
labels = [row['class_name'] for row in topk_rows]
values = [row['probability_percent'] for row in topk_rows]

plt.figure(figsize=(10, 4))
plt.bar(labels, values)
plt.ylabel('Confidence (%)')
plt.title(f'Top-{len(topk_rows)} Predictions')
plt.xticks(rotation=25, ha='right')
plt.tight_layout()
plt.show()